# Processed Data Playground
Create plots, visualize distributions and understand cohort selection

In [1]:
import pandas as pd
import os
import sys
import json
from pathlib import Path
import numpy as np

# Notebook is in notebooks/, so repo root is parent
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

In [3]:
from src.models.xgboost import load_data_files

# Get repo root relative to the current notebook
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Load static preprocessing config
config_path = os.path.join(repo_root, "configs", "xgboost_params.json")
with open(config_path, "r") as f:
    config = json.load(f)

# Set input and output directories
in_dir = os.path.join(repo_root, config["paths"]["in_dir"])


ed_vitals, clinical_encounters, ecg_records = load_data_files(in_dir, config)

In [8]:
print(ed_vitals.shape)
ed_vitals.head()

(1225239, 10)


,subject_id,stay_id,charttime,temperature,heartrate,resprate,o2sat,sbp,dbp,chartdate
0,17195991,38649090,2110-01-11 21:45:00,NaN,141.0,24.0,94.0,NaN,NaN,2110-01-11
1,17195991,38649090,2110-01-11 21:50:00,NaN,123.0,24.0,91.0,151.0,120.0,2110-01-11
2,17195991,38649090,2110-01-11 22:00:00,NaN,133.0,23.0,99.0,180.0,86.0,2110-01-11
3,17195991,38649090,2110-01-11 22:07:00,NaN,164.0,24.0,99.0,198.0,116.0,2110-01-11
4,17195991,38649090,2110-01-11 22:23:00,NaN,130.0,16.0,100.0,235.0,126.0,2110-01-11


In [9]:
print(clinical_encounters.shape)
clinical_encounters.head()

(494231, 19)


,subject_id,hadm_id,ed_stay_id,ed_intime,ed_outtime,hosp_admittime,hosp_dischtime,race,gender,anchor_age,death_time,icu_stay_id,icu_intime,icu_outtime,icu_count,diagnosis,icd_codes,diagnosis_labels,is_cardiovascular
0,10000032,22595853.0,33258284.0,2180-05-06 19:17:00,2180-05-06 23:30:00,2180-05-06 22:23:00,2180-05-07 17:15:00,WHITE,F,52.0,2180-09-09 00:00:00,NaN,NaN,NaN,NaN,"['Portal hypertension', 'Other ascites', 'Cirr...","['572.3', '789.59', '571.5', '070.70', '496', ...",[],False
1,10000032,22841357.0,38112554.0,2180-06-26 15:54:00,2180-06-26 21:31:00,2180-06-26 18:27:00,2180-06-27 18:49:00,WHITE,F,52.0,2180-09-09 00:00:00,NaN,NaN,NaN,NaN,['Unspecified viral hepatitis C with hepatic c...,"['070.71', '789.59', '287.5', '276.1', '496', ...",[],False
2,10000032,25742920.0,35968195.0,2180-08-05 20:58:00,2180-08-06 01:44:00,2180-08-05 23:44:00,2180-08-07 17:50:00,WHITE,F,52.0,2180-09-09 00:00:00,NaN,NaN,NaN,NaN,['Chronic hepatitis C without mention of hepat...,"['070.54', '789.59', 'V46.2', '571.5', '276.7'...",[],False
3,10000032,29079034.0,39399961.0,2180-07-23 05:54:00,2180-07-23 14:00:00,2180-07-23 12:35:00,2180-07-25 17:55:00,WHITE,F,52.0,2180-09-09 00:00:00,[39553978],['2180-07-23 14:00:00'],['2180-07-23 23:50:47'],1.0,"['Other iatrogenic hypotension', 'Chronic hepa...","['458.29', '070.44', '799.4', '276.1', '789.59...",[],False
4,10000117,27988844.0,33176849.0,2183-09-18 08:41:00,2183-09-18 20:20:00,2183-09-18 18:10:00,2183-09-21 16:30:00,WHITE,F,48.0,NaN,NaN,NaN,NaN,NaN,['Unspecified intracapsular fracture of left f...,"['S72.012A', 'W01.0XXA', 'Y93.K1', 'Y92.480', ...",['valvular_endocardial_disease'],True


In [14]:
ed_encounters = clinical_encounters[clinical_encounters["ed_stay_id"].notna()]
print("Number of ed encounters:", ed_encounters.shape[0])
print("Number of unique ED patients:", ed_encounters["subject_id"].nunique())

hosp_encounters = clinical_encounters[clinical_encounters["hadm_id"].notna()]
print("Number of hospital encounters:", hosp_encounters.shape[0])
print("Number of unique hospital patients:", hosp_encounters["subject_id"].nunique())


Number of ed encounters: 265702
Number of unique ED patients: 102125
Number of hospital encounters: 360268
Number of unique hospital patients: 124947


In [10]:
clinical_encounters['race'].value_counts()

race
WHITE                                        287659
BLACK/AFRICAN AMERICAN                        86835
OTHER                                         17244
HISPANIC/LATINO - PUERTO RICAN                13742
WHITE - OTHER EUROPEAN                        10749
BLACK/CAPE VERDEAN                             8121
HISPANIC OR LATINO                             7635
HISPANIC/LATINO - DOMINICAN                    7632
UNKNOWN                                        7270
WHITE - RUSSIAN                                7144
ASIAN - CHINESE                                6332
ASIAN                                          5556
BLACK/CARIBBEAN ISLAND                         3723
BLACK/AFRICAN                                  3502
HISPANIC/LATINO - GUATEMALAN                   2024
PORTUGUESE                                     1668
WHITE - EASTERN EUROPEAN                       1608
PATIENT DECLINED TO ANSWER                     1551
ASIAN - SOUTH EAST ASIAN                       1501
HISPANI

In [11]:
clinical_encounters['gender'].value_counts()

gender
F    262796
M    231435
Name: count, dtype: int64